# A4b — tham chiếu "app đã thấy" cho A4 (TASKS mục A4b; HUONG_DAN_GPU_08-11)
Chạy chính `q4b_holdout` (và `q4b_holdout_hist` nếu còn giờ) trên **132 mẫu dev dùng app đã thấy** (`holdout/dev_ids.txt`), chữ sạch, luật bật như A4, rồi tính khoảng cách với điểm app lạ của A4 (41,93 và 43,37). Khoảng 10 phút GPU cho suy luận, cộng thời gian khôi phục output A4.

**Trước khi chạy cell đầu tiên:** đặt notebook **Private**, Internet **Off**, Accelerator **GPU (RTX PRO 6000)**.
**Input cần gắn:** `lifemate-code` (bản mới từ commit `3ee6803` trở lên), `train+public_test` (raw), `thuan-qwen3-4b`, `phowhisper-large-thuan`, và **output của notebook A4** (`a4-unseen-apps`). **KHÔNG gắn `Hello_done`** hay output `train-hist`: adapter `q4b_holdout` nằm trong output của A4, và gắn hai output cùng lúc làm bước khôi phục chọn nhầm.
Cell B-1 kiểm tra sha256 hai adapter với giá trị đã ghi ở A4; sai thì dừng.

In [ ]:
%%bash
# P-1: setup (khôi phục state, checkpoints, outputs từ output của notebook A4)
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
QW=$(grep -rlE --include=config.json '"model_type"[[:space:]]*:[[:space:]]*"qwen3"' /kaggle/input 2>/dev/null | head -1 | xargs -r dirname)
PH=$(grep -rlE --include=config.json '"model_type"[[:space:]]*:[[:space:]]*"whisper"' /kaggle/input 2>/dev/null | head -1 | xargs -r dirname)
echo "Qwen3 dir  : ${QW:-KHÔNG THẤY}"; echo "Whisper dir: ${PH:-KHÔNG THẤY}"
[ -n "$QW" ] && [ -n "$PH" ] || { echo "ERROR: thiếu thư mục model"; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e "s#^PHO=.*#PHO=$PH#" -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục?"; ls /kaggle/working/checkpoints /kaggle/working/checkpoints/q4b_lora | head

In [ ]:
%%bash
# P-2: gỡ torchao, in phiên bản torch/CUDA/GPU (xác nhận cùng môi trường như A4)
pip uninstall -y torchao 2>&1 | tail -1
python - <<'EOF' | tee /kaggle/working/torch_cuda.txt
import sys
import torch
print(torch.__version__, torch.version.cuda, sys.version)
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__, "| gpus", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
if not torch.cuda.is_available():
    raise SystemExit("Phiên này KHÔNG có GPU: bật Accelerator rồi chạy lại")
EOF

In [ ]:
%%bash
# P-3: ghi sẵn các đoạn mã phụ (chỉ in số đếm tổng hợp, không in nội dung dữ liệu)
mkdir -p /kaggle/working/helpers
cat > /kaggle/working/helpers/fr.py <<'PYEOF'
"""Chỉ in SỐ ĐẾM tổng hợp, không in nội dung câu nói hay dự đoán.
usage: python fr.py <ids_file> <submission.jsonl> [<submission.jsonl> ...]"""
import json
import sys
from pathlib import Path

ids = Path(sys.argv[1]).read_text().split()
gold = {}
with open("data/raw/train+public_test/train/train.jsonl", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        gold[d["id"]] = d["label"]["tool_calls"]


def load(path):
    with open(path, encoding="utf-8") as f:
        return {json.loads(line)["id"]: json.loads(line)["tool_calls"] for line in f}


def is_ask(calls):
    return any(c["name"] == "sys.ask_missing_info" for c in calls)


n_gold_call = sum(1 for i in ids if gold[i])
n_oos = sum(1 for i in ids if not gold[i])
print(f"{len(ids)} mẫu: {n_gold_call} có call, {n_oos} ngoài phạm vi (oos)")
print(f"{'file':44} {'từ chối nhầm':>14} {'oos đúng':>10} {'dự đoán ask':>12} {'call TB':>8}")
for p in sys.argv[2:]:
    if not Path(p).exists():
        print(f"{p:44} (không có file)")
        continue
    r = load(p)
    refused = sum(1 for i in ids if gold[i] and not r.get(i))
    oos_ok = sum(1 for i in ids if not gold[i] and not r.get(i))
    asks = sum(1 for i in ids if r.get(i) and is_ask(r[i]))
    avg = sum(len(r.get(i, [])) for i in ids) / len(ids)
    print(f"{Path(p).name:44} {refused:>7}/{n_gold_call:<6} {oos_ok:>5}/{n_oos:<4} {asks:>12} {avg:>8.2f}")
PYEOF
cat > /kaggle/working/helpers/pack_zip.py <<'PYEOF'
"""Đóng gói file kết quả nhỏ thành một zip kèm MANIFEST sha256 (không đóng gói trọng số).
usage: python pack_zip.py <out.zip> <file> [<file> ...]   (file không tồn tại thì bỏ qua và báo)"""
import hashlib
import sys
import zipfile
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()


out = Path(sys.argv[1])
files = []
for name in sys.argv[2:]:
    p = Path(name)
    if p.is_file():
        files.append(p)
    else:
        print("bỏ qua (không có):", name)
files = sorted(set(files), key=str)
manifest = "".join(f"{sha256(p)}  {p.as_posix()}  ({p.stat().st_size} bytes)\n" for p in files)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.as_posix().lstrip("/"))
    z.writestr("MANIFEST_sha256.txt", manifest)
print(f"{out}: {out.stat().st_size / 1e6:.2f} MB, {len(files)} file")
PYEOF
cat > /kaggle/working/helpers/gap.py <<'PYEOF'
"""Khoảng cách 'app đã thấy' - 'app lạ' của CÙNG MỘT model. Chỉ in số tổng hợp (không in nội dung câu nói/dự đoán).
usage: python gap.py <seen_ids> <seen.jsonl> <unseen_ids> <unseen.jsonl> [<tên>]
Chạy từ thư mục gốc repo (cần scripts/compare_runs.py, data/raw/..., data/processed/gold_hash_train.json)."""
import json
import random
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "src"))
import compare_runs as cr  # noqa: E402
from lifemate.metric import load_fuzzy_params  # noqa: E402

seen_ids, seen_sub, unseen_ids, unseen_sub = sys.argv[1:5]
name = sys.argv[5] if len(sys.argv) > 5 else "model"
all_items = [json.loads(line) for line in open(cr.DATA / "train.jsonl", encoding="utf-8")]
gh = json.loads((ROOT / "data" / "processed" / "gold_hash_train.json").read_text(encoding="utf-8"))
fuzzy = load_fuzzy_params(cr.DATA / "train_tools_catalog.json")


def scores(ids_file, sub):
    keep = set(Path(ids_file).read_text().split())
    items = [i for i in all_items if i["id"] in keep]
    s = cr.per_sample(sub, items, gh, fuzzy)
    return {i["id"]: (cr.kind(i["label"]["tool_calls"]), s[i["id"]]) for i in items}


def mean(vals):
    vals = list(vals)
    return 100 * sum(vals) / len(vals) if vals else float("nan")


def boot_gap(a, b, n=5000, seed=0):
    """Khoảng tin cậy 95% của mean(a) - mean(b) cho HAI tập mẫu độc lập (bootstrap từng tập)."""
    rng = random.Random(seed)
    gaps = sorted(
        sum(a[rng.randrange(len(a))] for _ in a) / len(a) - sum(b[rng.randrange(len(b))] for _ in b) / len(b) for _ in range(n)
    )
    return 100 * gaps[int(0.025 * n)], 100 * gaps[int(0.975 * n)]


seen, unseen = scores(seen_ids, seen_sub), scores(unseen_ids, unseen_sub)
print(f"=== {name}: app ĐÃ THẤY (n={len(seen)}) so với app LẠ (n={len(unseen)}), chữ sạch, luật bật")
print(f"{'nhóm':28} {'đã thấy':>10} {'(n)':>5} {'lạ':>8} {'(n)':>5} {'khoảng cách':>12}   khoảng tin cậy 95%")


def row(label, pick):
    a = [v for k, v in seen.values() if pick(k)]
    b = [v for k, v in unseen.values() if pick(k)]
    if not a or not b:
        print(f"{label:28} {mean(a):>10.2f} {len(a):>5} {mean(b):>8.2f} {len(b):>5} {'(thiếu một bên)':>12}")
        return None
    lo, hi = boot_gap(a, b)
    gap = mean(a) - mean(b)
    print(f"{label:28} {mean(a):>10.2f} {len(a):>5} {mean(b):>8.2f} {len(b):>5} {gap:>+12.2f}   [{lo:+.2f}, {hi:+.2f}]")
    return gap


g_total = row("TỔNG (cách TASKS tính)", lambda k: True)
g_noos = row("chỉ exec + ask (cùng loại)", lambda k: k != "oos")
g_exec = row("exec", lambda k: k == "exec")
g_ask = row("ask", lambda k: k == "ask")
seen_oos = [v for k, v in seen.values() if k == "oos"]
print(f"{'oos (chỉ có ở đã thấy)':28} {mean(seen_oos):>10.2f} {len(seen_oos):>5}")
print()
print("Ngưỡng của Phương: dưới ~6 điểm thì bỏ A5, trên ~15 điểm thì làm A5, ở giữa thì báo lại.")
print("Lưu ý: 'TỔNG' có thêm các mẫu oos ở phía đã thấy (dễ điểm), không có ở phía app lạ, nên thổi phồng khoảng cách;")
print("khoảng cách 'chỉ exec + ask' so cùng loại mẫu nên đáng tin hơn. Hai tập mẫu khác nhau, khoảng tin cậy là của hai mẫu độc lập.")
for label, g in (("TỔNG", g_total), ("exec + ask", g_noos)):
    if g is not None:
        verdict = "dưới 6: bỏ A5" if g < 6 else ("trên 15: làm A5" if g > 15 else "ở giữa 6 và 15: báo lại")
        print(f"  - {label}: khoảng cách {g:+.2f} -> {verdict}")
PYEOF
ls -la /kaggle/working/helpers

In [ ]:
%%bash
# B-1: kiểm tra đủ đồ và đúng adapter của A4. Thiếu hoặc sai sha256 thì DỪNG
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
MISSING=0
need() { if [ -s "$1" ]; then echo "ok     $1"; else echo "THIẾU  $1"; MISSING=1; fi; }
need scripts/compare_runs.py
need data/processed/gold_hash_train.json
need data/processed/holdout/dev_ids.txt
need data/processed/holdout/eval_ids.txt
need checkpoints/q4b_holdout/adapter_model.safetensors
need checkpoints/q4b_holdout_hist/adapter_model.safetensors
need outputs/unseen_q4b_holdout.jsonl
need outputs/unseen_q4b_holdout_hist.jsonl
grep -q build_run_manifest src/lifemate/manifest.py 2>/dev/null && echo "ok     code là bản mới (có build_run_manifest)" || { echo "THIẾU  code còn là bản CŨ: upload lifemate-code mới"; MISSING=1; }
if [ "$MISSING" = 1 ]; then echo "-> Gắn output notebook A4 (a4-unseen-apps), dùng lifemate-code mới, và KHÔNG gắn Hello_done."; exit 1; fi
echo; echo "== sha256 adapter (phải trùng với A4)"
h1=$(sha256sum checkpoints/q4b_holdout/adapter_model.safetensors | cut -d' ' -f1)
h2=$(sha256sum checkpoints/q4b_holdout_hist/adapter_model.safetensors | cut -d' ' -f1)
echo "q4b_holdout      $h1"; echo "q4b_holdout_hist $h2"
[ "$h1" = "3fb678f340662f002a6bb4d42c28f91848a6d631839c1d9ff5b900bf33199a0a" ] && echo "q4b_holdout: TRÙNG" || { echo "q4b_holdout: KHÔNG TRÙNG, dừng"; exit 1; }
[ "$h2" = "d425d17c31ce87e87515b2356ba6402c1943de4919b20cd89b57b2569c8b0910" ] && echo "q4b_holdout_hist: TRÙNG" || { echo "q4b_holdout_hist: KHÔNG TRÙNG, dừng"; exit 1; }
echo; echo "== số mẫu"; wc -l data/processed/holdout/dev_ids.txt data/processed/holdout/eval_ids.txt
echo; df -h /kaggle/working | tail -1
mkdir -p outputs/logs

In [ ]:
%%bash
# B-2: dọn BẢN SAO khôi phục trong phiên để output nhỏ (bản gốc vẫn nằm nguyên trong input là output notebook A4).
# Giữ adapter cuối của q4b_holdout và q4b_holdout_hist; bỏ các checkpoint-* trung gian và các adapter không dùng ở A4b.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
du -sh checkpoints | cut -f1 | xargs echo "trước khi dọn:"
rm -rf checkpoints/phowl_ft checkpoints/q4b_lora checkpoints/q4b_hist
rm -rf checkpoints/q4b_holdout/checkpoint-* checkpoints/q4b_holdout_hist/checkpoint-*
du -sh checkpoints | cut -f1 | xargs echo "sau khi dọn:"
ls checkpoints; df -h /kaggle/working | tail -1

In [ ]:
%%bash
# B-3: q4b_holdout trên 132 mẫu dev dùng app ĐÃ THẤY (chữ sạch, luật bật, đúng lệnh của TASKS A4b)
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
bash scripts/infer_multi_gpu.sh train outputs/seen_q4b_holdout.jsonl --ids $H/dev_ids.txt --backend hf --model $QWEN --adapter checkpoints/q4b_holdout --text clean
python -X utf8 scripts/eval.py --sub outputs/seen_q4b_holdout.jsonl --ids $H/dev_ids.txt | tee outputs/logs/a4b_q4b_holdout_eval.txt
tail -qn 1 outputs/seen_q4b_holdout.part*.log | tee outputs/logs/a4b_q4b_holdout_timing.txt

In [ ]:
%%bash
# B-4: q4b_holdout_hist trên cùng tệp. Thiếu giờ GPU thì đặt DO_HIST=0
DO_HIST=1
[ "$DO_HIST" = 1 ] || { echo "bỏ qua q4b_holdout_hist"; exit 0; }
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
bash scripts/infer_multi_gpu.sh train outputs/seen_q4b_holdout_hist.jsonl --ids $H/dev_ids.txt --backend hf --model $QWEN --adapter checkpoints/q4b_holdout_hist --text clean
python -X utf8 scripts/eval.py --sub outputs/seen_q4b_holdout_hist.jsonl --ids $H/dev_ids.txt | tee outputs/logs/a4b_q4b_holdout_hist_eval.txt
tail -qn 1 outputs/seen_q4b_holdout_hist.part*.log | tee outputs/logs/a4b_q4b_holdout_hist_timing.txt

In [ ]:
%%bash
# B-5: KẾT QUẢ CHÍNH: khoảng cách đã thấy - lạ (cùng một model), so cặp hist với mốc trên app đã thấy, đếm từ chối nhầm
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
H=data/processed/holdout
{
echo "################ q4b_holdout: đã thấy so với lạ"
python -X utf8 /kaggle/working/helpers/gap.py $H/dev_ids.txt outputs/seen_q4b_holdout.jsonl $H/eval_ids.txt outputs/unseen_q4b_holdout.jsonl q4b_holdout
echo
if [ -s outputs/seen_q4b_holdout_hist.jsonl ]; then
  echo "################ q4b_holdout_hist: đã thấy so với lạ"
  python -X utf8 /kaggle/working/helpers/gap.py $H/dev_ids.txt outputs/seen_q4b_holdout_hist.jsonl $H/eval_ids.txt outputs/unseen_q4b_holdout_hist.jsonl q4b_holdout_hist
  echo
  echo "################ So từng cặp trên app ĐÃ THẤY: q4b_holdout -> q4b_holdout_hist"
  python -X utf8 scripts/compare_runs.py --a outputs/seen_q4b_holdout.jsonl --b outputs/seen_q4b_holdout_hist.jsonl --ids $H/dev_ids.txt --names q4b_holdout,q4b_holdout_hist
  echo
fi
echo "################ Từ chối nhầm và cơ cấu dự đoán trên app đã thấy"
python -X utf8 /kaggle/working/helpers/fr.py $H/dev_ids.txt outputs/seen_q4b_holdout.jsonl outputs/seen_q4b_holdout_hist.jsonl
} 2>&1 | tee outputs/logs/a4b_report.txt

In [ ]:
%%bash
# B-6: gói kết quả nhỏ để gửi Phương: a4b_results.zip (điểm, so sánh, dự đoán seen_*.jsonl, log, torch_cuda.txt)
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 /kaggle/working/helpers/pack_zip.py /kaggle/working/a4b_results.zip outputs/seen_q4b_holdout.jsonl outputs/seen_q4b_holdout_hist.jsonl outputs/logs/a4b_*.txt /kaggle/working/torch_cuda.txt
du -sh /kaggle/working/checkpoints /kaggle/working/outputs; df -h /kaggle/working | tail -1

In [ ]:
%%bash
# B-7 (CUỐI CÙNG): lưu trạng thái rồi Save Version. Cell này xóa thư mục làm việc lifemate/.
bash /kaggle/working/lifemate/scripts/kaggle_save_state.sh
du -sh /kaggle/working/state /kaggle/working/checkpoints /kaggle/working/outputs